# Project Overview

This project explores a Spotify dataset containing over 114,000 tracks and their audio features. The objective is to clean the dataset and perform exploratory data analysis (EDA) in order to better understand patterns in song popularity, genre distribution, track duration, and various audio characteristics. The analysis aims to identify trends and insights that can help explain how songs differ across genres and popularity levels.

A summary of the project, research questions, visualizations, and key findings is presented in the accompanying portfolio website.

# Cleaning the Dataset

First, the the dataset must be inspected and prepared before conducting any analysis. Data cleaning helps ensure that the results are based on accurate, consistent, and reliable data. In this section, the dataset is examined for missing values, duplicate records, incorrect data types, and potential inconsistencies.

## Introduction to the Dataset

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshanaasthigiri/spotify-tracks-dataset/spotify-tracks-dataset-detailed.csv


In [ ]:
df = pd.read_csv("/kaggle/input/datasets/lakshanaasthigiri/spotify-tracks-dataset/spotify-tracks-dataset-detailed.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Checking Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
df[df.isnull().any(axis = 1)]

In [ ]:
missing_percent = (df.isnull() / len(df)) * 100
missing_percent

In [ ]:
df = df.dropna()

In [ ]:
df

In [ ]:
df.isnull().sum()

## Checking for Duplicate Rows

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
df = df.drop_duplicates()

In [ ]:
df

In [ ]:
df.duplicated().sum()

## Checking for Duplicate Track ID's

In [ ]:
df["track_id"].duplicated().sum()

In [ ]:
duplicate_tracks = df[df["track_id"].duplicated(keep=False)]
duplicate_tracks[["track_id", "track_name", "artists", "track_genre"]].head(20)

## Checking Column Names & Data Types

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.columns

In [ ]:
df.dtypes

Converting Duration to Minutes

In [ ]:
df["duration_min"] = df["duration_ms"] / 60000

In [ ]:
df[["duration_ms", "duration_min"]].head()

## Checking Duration Outliers

In [ ]:
df["duration_min"].describe()

In [ ]:
df.nsmallest(10, "duration_min")[
    ["track_name", "artists", "duration_min"]
]

In [ ]:
df.nlargest(10, "duration_min")[
    ["track_name", "artists", "duration_min"]
]

## Checking Feature Ranges

In [ ]:
audio_features = ["danceability", "energy", "speechiness", "acousticness", "instrumentalness", "liveness", "valence"]

for feature in audio_features:
    print(feature, df[feature].min(), df[feature].max())

## Checking Popularity

In [ ]:
df["popularity"].describe()

In [ ]:
df[
    (df["popularity"] < 0) |
    (df["popularity"] > 100)
]

## Checking Tempo

In [ ]:
df["tempo"].describe()

In [ ]:
df.nlargest(10, "tempo")[
    ["track_name", "tempo"]
]

## Checking Genres

In [ ]:
df["track_genre"].nunique()

In [ ]:
df["track_genre"].value_counts()

# Exploratory Data Analysis

After cleaning the dataset, exploratory data analysis is performed to better understand the characteristics of Spotify tracks. This stage focuses on identifying patterns, distributions, relationships, and potential trends within the data using summary statistics and visualizations.

## What does this dataset contain?

In [ ]:
print(f"Number of songs: {len(df):,}")
print(f"Number of artists: {df['artists'].nunique():,}")
print(f"Number of albums: {df['album_name'].nunique():,}")
print(f"Number of genres: {df['track_genre'].nunique():,}")

## Are most Spotify songs popular?

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df["popularity"], bins=20)
plt.title("Distribution of Spotify Song Popularity")
plt.xlabel("Popularity")
plt.ylabel("Number of Songs")
plt.show()

In [ ]:
df["popularity"].describe()

## What genres have the most songs?

In [ ]:
genre_popularity = (
    df.groupby("track_genre")["popularity"]
      .mean()
      .sort_values(ascending=False)
      .head(15)
)

plt.figure(figsize=(10,6))
genre_popularity.sort_values().plot(kind="barh")
plt.title("Top 15 Genres by Average Popularity")
plt.xlabel("Average Popularity")
plt.show()

## How long are Spotify songs typically?

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["duration_min"], bins=40)
plt.xlim(0,10)
plt.title("Distribution of Song Duration")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of Songs")
plt.show()

In [ ]:
df["duration_min"].describe()

## What are the distributions for features such as danceability, energy, valence, acousticness, and speechiness?

In [ ]:
features = ["danceability", "energy", "valence", "acousticness", "speechiness"]

for feature in features:
    plt.figure(figsize=(6,4))
    plt.hist(df[feature], bins=30)
    plt.title(f"Distribution of {feature.capitalize()}")
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.show()

## What numerical features are related?

In [ ]:
import seaborn as sns

plt.figure(figsize=(12,10))

corr = df.select_dtypes(include="number").corr()

sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
audio_features = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo"]

popularity_corr = df[audio_features + ["popularity"]].corr()["popularity"].sort_values(ascending=False)

popularity_corr = popularity_corr.drop("popularity")

popularity_corr

## Do popular songs and less popular songs differ in their audio characteristics?

In [ ]:
df["popularity_group"] = df["popularity"].apply(lambda x: "Popular" if x > 35 else "Less Popular")

group_means = df.groupby("popularity_group")[audio_features].mean()

group_means

## Do explicit songs differ from non-explicit songs?

In [ ]:
df.groupby("explicit")["popularity"].mean().plot(kind="bar")

plt.title("Average Popularity by Explicit Content")
plt.ylabel("Average Popularity")
plt.show()

## Who are the most popular artists?

In [ ]:
top_artists = (
    df.groupby("artists")["popularity"]
      .mean()
      .sort_values(ascending=False)
      .head(15)
)

top_artists.sort_values().plot(kind="barh", figsize=(10,6))

plt.title("Artists with Highest Average Popularity")
plt.xlabel("Average Popularity")
plt.show()

This notebook demonstrated the process of cleaning and exploring a large Spotify dataset through exploratory data analysis. The insights and key findings from this analysis are summarized separately as part of the accompanying portfolio project.